In [2]:
import re
spanclasses = {}

def addentry(curdict,curitem):
    if(curitem in curdict):
        curdict[curitem] = curdict[curitem] + 1
    else:
        curdict[curitem] = 1

def heikchap(bnum,outf):
    inh2 = 0
    inlg = 0
    curchap = 0
    curpara = 0
    curbooks = ''
    global spanclasses
    fname = '/Users/gcrane/scratch/heike-japanese/Heike' + bnum + '.xml'

    print(fname)
    fname = re.sub('(Heike)([0-9]\.xml)','\g<1>0\g<2>',fname)
    f = open(fname)
    noten = 0
    for l in f:
        l = re.sub('\s+$','',l)
        if(re.search('<h2 ',l)):
            inh2 = 1
        if(re.search("tei_trailer",l)):
            inh2 = 0
            continue
        if(not inh2):
            continue

        m = re.search('<h2 class="normal">([0-9]+)([^<]+)</h2>',l)
        if(m):
            curbooks = m[1]
            l = re.sub('<h2 class="normal">([0-9]+)([^<]+)</h2>','<div type="textpart" subtype="book" n="\g<1>"><head>\g<1>\g<2></head>',l)
        
        if(re.search('<div class="tei_lg">',l)):
            inlg = 1
            print('<lg>',file=outf)
            continue
        if(re.search('</div>',l) and inlg):
            inlg = 0
            print('</lg>',file=outf)
            continue

        m = re.search('<h4 class="normal">(.+)</h4>',l)
        if(m):
            curchap = curchap + 1
            curpara = 0
            curchaps = str(curchap)

            if(not curchap == 1):
                print('</div>',file=outf)
            xmlbase = 'xml:base="urn:cts:japaneseLit:heike.tokyo1933.perseus-jpn1:' + curbooks + '"'
            l = re.sub('<h4 class="normal">(.+)</h4>','<div type="textpart" subtype="chapter" '+xmlbase+' n="'+str(curchap)+'"><head>'+curbooks+'.'+str(curchap)+'. \g<1></head>',l)


        l = re.sub('</p>','</s></p>\n</div>',l)
            
        l = re.sub('<span>([^<]+)\s*</span><br />','<l>\g<1></l>',l)
        l = re.sub('<span class="tei_italics">([^<]+)</span>','<title>\g<1></title>',l)
        l = re.sub('<span class="tei_small">([^<]+)</span>','<hi rend="tei_small">\g<1></hi>',l)
        l = re.sub('<span class="tei_kanasub">([^<]+)</span>','<foreign xml:lang="tei_kanasub">\g<1></foreign>',l)
        l = re.sub('<div><a name="([^"]+)"></a><div class="tei_note_number">([^<]+)</div>(.+)</div>','<note xml:id="\g<1>">\g<2> \g<3></note>',l)
        while(re.search('<div><a name=""></a><div class="tei_note_number">([^<]+)</div>([^>]+)</div>',l)):
            m = re.search('<div><a name=""></a><div class="tei_note_number">([^<]+)</div>([^>]+)</div>',l)
            if(m):
                noten = noten + 1
                curnote = bnum + '.' + str(noten)
                print('curnote',curnote)
                l = re.sub('<div><a name=""></a><div class="tei_note_number">([^<]+)</div>([^>]+)</div>','<ref target="'+curnote+'">\g<1></ref>\g<2>',l,1)
        m = re.search('<span class="([^"]+)"',l)
        if(m):
            print(m[1])
            addentry(spanclasses,m[1])

        if(re.search('<p class="tei_p">',l)):
            curpara = curpara + 1
            cursentid = 0
            curparas = str(curpara)
            curcit = '[' + curbooks + '.' + curchaps + '.' + curparas + '] '
            curcit2 = curbooks + '.' + curchaps + '.' + curparas
            xmlbase = 'xml:base="urn:cts:japaneseLit:heike.tokyo1933.perseus-jpn1:' + curbooks + '.' + curchaps + '"' 

            l = re.sub('<p class="tei_p">','<div type="textpart" subtype="section" '+xmlbase+' n="'+curparas+'"><p>'+curcit + '<s>',l)
            if(curpara > 1 and 0 ):
                l = '</div>\n' + l
        l = re.sub('<s>\s*</s>','',l)
        if(re.search('(<s>|</s>)',l)):
            l = re.sub('。','。</s> <s>',l)

        if(re.search('<span[^>]+>',l) and not re.search('<span[^>]+>[^<]+</span>',l)):
            print('span',l)
        while(re.search('(<s>|<l>)',l)):
            m = re.search('(<s>|<l>)',l)
            cursentid = cursentid + 1
            curid = curcit2 + '.' + str(cursentid)
            l = re.sub('<([ls])>','<\g<1> xml:id="sent'+curid+'">',l,1)
        l = re.sub('</s>\s<s','</s>\n<s',l)
        print(l,file=outf)

    print('</div>',file=outf)
    print('</div>',file=outf)
    f.close()

outf = open('/Users/gcrane/github/GRC_misc/heike-tokyo1933-jpn1.xml','w')
f = open('/Users/gcrane/scratch/heike-japanese/heike-teiheader.xml')
textb = f.read()
f.close()
outf.write(textb)
for i in range(1,13):
    heikchap(str(i),outf)

print('</div>\n</body>\n</text>\n</TEI>\n',file=outf)
outf.close()

print('\ntots')
for foo in spanclasses:
    print(foo,spanclasses[foo])

/Users/gcrane/scratch/heike-japanese/Heike1.xml
curnote 1.1
curnote 1.2
curnote 1.3
/Users/gcrane/scratch/heike-japanese/Heike2.xml
curnote 2.1
curnote 2.2
/Users/gcrane/scratch/heike-japanese/Heike3.xml
/Users/gcrane/scratch/heike-japanese/Heike4.xml
curnote 4.1
/Users/gcrane/scratch/heike-japanese/Heike5.xml
/Users/gcrane/scratch/heike-japanese/Heike6.xml
/Users/gcrane/scratch/heike-japanese/Heike7.xml
curnote 7.1
curnote 7.2
curnote 7.3
curnote 7.4
curnote 7.5
curnote 7.6
/Users/gcrane/scratch/heike-japanese/Heike8.xml
curnote 8.1
curnote 8.2
/Users/gcrane/scratch/heike-japanese/Heike9.xml
curnote 9.1
curnote 9.2
curnote 9.3
curnote 9.4
/Users/gcrane/scratch/heike-japanese/Heike10.xml
curnote 10.1
curnote 10.2
curnote 10.3
/Users/gcrane/scratch/heike-japanese/Heike11.xml
curnote 11.1
curnote 11.2
curnote 11.3
curnote 11.4
curnote 11.5
/Users/gcrane/scratch/heike-japanese/Heike12.xml
curnote 12.1

tots
